# 🤖 AI Resume Screening System
### Data Science Internship — GenAI Assignment
**Stack:** LangChain · OpenAI · LangSmith Tracing

---
**Pipeline:** `Resume → Skill Extraction → Matching → Scoring → Explanation → Tracing`

## 1. Setup & Environment

In [ ]:
# Install dependencies (run once)
# !pip install -r requirements.txt

import os
import json
from dotenv import load_dotenv

load_dotenv()  # Loads .env file

# Verify tracing is enabled
print('LANGCHAIN_TRACING_V2 :', os.getenv('LANGCHAIN_TRACING_V2'))
print('LANGCHAIN_PROJECT    :', os.getenv('LANGCHAIN_PROJECT'))
print('OpenAI key loaded    :', bool(os.getenv('OPENAI_API_KEY')))

## 2. Load Sample Data

In [ ]:
from resumes.sample_data import CANDIDATES, JOB_DESCRIPTION

print(f'Candidates loaded: {list(CANDIDATES.keys())}')
print(f'\n--- JOB DESCRIPTION PREVIEW ---')
print(JOB_DESCRIPTION[:400], '...')

## 3. Inspect the Prompt Templates

In [ ]:
from prompts.templates import SKILL_EXTRACTION_TEMPLATE, SCORING_TEMPLATE

print('=== SKILL EXTRACTION PROMPT ===')
print(SKILL_EXTRACTION_TEMPLATE.template[:600])
print('\n=== SCORING PROMPT ===')
print(SCORING_TEMPLATE.template[:600])

## 4. Run the Full Screening Pipeline

In [ ]:
from chains.screening_chain import ResumeScreeningPipeline
from langsmith import traceable

# Instantiate pipeline
pipeline = ResumeScreeningPipeline(temperature=0.0)

@traceable(
    name='resume_screening_run',
    tags=['notebook', 'assignment', 'data-science-internship']
)
def screen_candidate(label, resume_text):
    return pipeline.run(
        resume_text=resume_text,
        job_description=JOB_DESCRIPTION,
        candidate_label=label
    )

results = {}
for label, resume in CANDIDATES.items():
    result = screen_candidate(label, resume)
    results[label] = result
    print(f'\n✅ {label} → Score: {result.score}/100 | {result.tier}')

## 5. View Detailed Results

In [ ]:
# Inspect strong candidate
strong = results['Strong Candidate – Priya Sharma']

print('=== EXTRACTED PROFILE ===')
print(json.dumps(strong.resume_profile, indent=2))

print('\n=== MATCHING ANALYSIS ===')
print(json.dumps(strong.matching_analysis, indent=2))

print('\n=== SCORE RESULT ===')
print(json.dumps(strong.score_result, indent=2))

print('\n=== EXPLANATION ===')
print(json.dumps(strong.explanation, indent=2))

In [ ]:
# Inspect weak candidate
weak = results['Weak Candidate – Amit Gupta']

print('=== WEAK CANDIDATE EXPLANATION ===')
print(json.dumps(weak.explanation, indent=2))

## 6. Candidate Ranking

In [ ]:
print('\n🏆 RANKING')
sorted_r = sorted(results.values(), key=lambda r: r.score, reverse=True)
for i, r in enumerate(sorted_r, 1):
    bar = '█' * (r.score // 5) + '░' * (20 - r.score // 5)
    print(f'#{i} {r.candidate_name:<30} {r.score:>3}/100  [{bar}]  {r.tier}')

## 7. LangSmith Tracing — Debugging an Incorrect Output

> 📌 **Task:** Demonstrate debugging by intentionally causing a bad output and fixing it.

We will pass a malformed resume (missing all skills) to show what happens in the pipeline,
then view the trace in LangSmith to identify where it broke.

In [ ]:
# ── Debug Run: Intentionally bad / incomplete resume ─────────────────────────
MALFORMED_RESUME = """
Name: Test Person
I am interested in working somewhere. I like computers and stuff.
"""

@traceable(
    name='debug_run_malformed_resume',
    tags=['debug', 'error-analysis', 'langsmith-demo']
)
def debug_malformed():
    return pipeline.run(
        resume_text=MALFORMED_RESUME,
        job_description=JOB_DESCRIPTION,
        candidate_label='DEBUG – Malformed Resume'
    )

debug_result = debug_malformed()

print('DEBUG RESULT:')
print(f'  Score      : {debug_result.score}/100')
print(f'  Tier       : {debug_result.tier}')
print(f'  Profile    : {json.dumps(debug_result.resume_profile, indent=2)}')
print('\n✅ Even with a bad resume, the pipeline did NOT hallucinate skills.')
print('   Check LangSmith for the full trace of this debug run.')

### LangSmith Observation
- Navigate to your LangSmith project: `https://smith.langchain.com`
- You should see **4 runs** (3 normal + 1 debug)
- Each run shows all 5 pipeline steps as nested spans
- The debug run shows the LLM correctly returning empty lists (no hallucination)